# Phase 2 — Feature Engineering

**Purpose:** Transform raw transaction data into a customer-level panel ready for modelling.

We construct:
- **Churn flag:** no purchase in the final 90 days of the observation window
- **CLV (Customer Lifetime Value):** total revenue per customer over the observation period
- **RFM features:** Recency (days since last purchase), Frequency (invoices per month), Monetary (total revenue)
- **Segment encoding:** integer-coded country, with rare countries collapsed into "Other" and then dropped

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.bcs.data import (
    load_raw,
    drop_missing_customers,
    drop_cancellations,
    drop_negative_quantity,
    drop_negative_price,
)
from src.bcs.features import build_customer_panel, log1p_scale

print("Imports loaded successfully.")

Imports loaded successfully.


## 1. Load and clean raw data

In [2]:
df = load_raw("../data/online_retail_II.xlsx")
print(f"Raw data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

Raw data loaded: 1,067,371 rows, 8 columns
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


In [3]:
df, missing_rate = drop_missing_customers(df)
print(f"Dropped {missing_rate:.1%} of rows with missing Customer ID")
print(f"Remaining: {df.shape[0]:,} rows")

Dropped 22.8% of rows with missing Customer ID
Remaining: 824,364 rows


In [4]:
cancellation_mask = df["Invoice"].astype(str).str.startswith("C")
cancellation_rate = cancellation_mask.mean()
n_cancelled = cancellation_mask.sum()
print(f"Cancelled invoices: {n_cancelled:,} ({cancellation_rate:.1%} of transactions)")
print("Decision: dropping cancellations as they reverse revenue and distort CLV.")

Cancelled invoices: 18,744 (2.3% of transactions)
Decision: dropping cancellations as they reverse revenue and distort CLV.


In [5]:
before = df.shape[0]
df = drop_cancellations(df)
df = drop_negative_quantity(df)
df = drop_negative_price(df)
after = df.shape[0]
print(f"Removed {before - after:,} rows ({(before - after) / before:.1%})")
print(f"Clean data: {after:,} rows, {df.shape[1]} columns")

Removed 18,815 rows (2.3%)
Clean data: 805,549 rows, 8 columns


## 2. Build customer panel

Constructs all features in one step:
- **Churn flag:** no purchase in last 90 days
- **CLV:** total revenue over observation window
- **RFM:** recency, frequency, monetary value
- **Segment encoding:** country grouped, rare countries collapsed into "Other" and then dropped

**Dropping "Other":** The "Other" segment aggregates genuinely different countries (Channel Islands, Norway, Austria, Denmark, Cyprus, Japan, USA, etc.) into a single pseudo-segment. These countries have nothing in common except being small, so the "Other" estimate sits near the global mean anyway, making partial pooling look redundant. Dropping them (138 customers, 2.3% of identified customers) gives a cleaner interpretation.

In [ ]:
panel, n_segments, n_other = build_customer_panel(
    df,
    churn_window_days=90,
    segment_col="Country",
    min_segment_size=15,
    drop_other=True,
)
print(f"Panel built: {len(panel):,} customers across {n_segments} segments")
print(f"Dropped {n_other} customers in 'Other' segment ({n_other / (len(panel) + n_other):.1%})")
print(f"Columns: {panel.columns.tolist()}")

## 3. Inspect segment-level heterogeneity

In [ ]:
seg_summary = panel.groupby("segment_name").agg(
    n_customers=("Customer ID", "count"),
    churn_rate=("churned", "mean"),
    mean_clv=("clv", "mean"),
    median_clv=("clv", "median"),
).sort_values("n_customers", ascending=False)
print(seg_summary.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
churn_by_seg = panel.groupby("segment_name")["churned"].mean().sort_values(ascending=False)
churn_by_seg.plot(kind="bar", ax=ax, color="steelblue")
ax.set_ylabel("Churn rate")
ax.set_title("Churn rate by country segment")
overall_rate = panel["churned"].mean()
ax.axhline(overall_rate, color="red", linestyle="--",
           label=f"Overall ({overall_rate:.1%})")
ax.legend()
plt.tight_layout()
plt.show()
spread_pp = (churn_by_seg.max() - churn_by_seg.min()) * 100
print(f"Churn rate range: {churn_by_seg.min():.1%} to {churn_by_seg.max():.1%}")
print(f"Spread: {spread_pp:.0f} percentage points")
if spread_pp < 5:
    print("NOTE: Spread < 5pp — heterogeneity is low. Consider CLV decile as alternative grouping.")
else:
    print(f"Spread > 5pp — meaningful heterogeneity detected. Partial pooling should add value.")
lowest = churn_by_seg.nsmallest(2)
highest = churn_by_seg.nlargest(2)
print(f"Lowest churn: {', '.join(lowest.index)} ({lowest.values[0]:.1%}, {lowest.values[1]:.1%})")
print(f"Highest churn: {', '.join(highest.index)} ({highest.values[0]:.1%}, {highest.values[1]:.1%})")

## 4. Inspect CLV distribution

Checking for skew in CLV. If skewness exceeds 5, we probably want to use log1p_scale for any specification with CLV as covariate. For CLV as outcome, modelling log(CLV) rather than raw CLV should be preferable.

In [ ]:
print(panel["clv"].describe())
print(f"Skewness: {panel['clv'].skew():.1f}")
if panel["clv"].skew() > 5:
    print("Skewness > 5 — use log1p transform for CLV in modelling.")
else:
    print("Skewness moderate — raw CLV may be usable but log transform is still recommended.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
panel["clv"].hist(bins=50, ax=axes[0])
axes[0].set_title("CLV distribution (raw)")
axes[0].set_xlabel("Total revenue")

np.log1p(panel["clv"]).hist(bins=50, ax=axes[1])
axes[1].set_title("CLV distribution (log1p)")
axes[1].set_xlabel("log1p(total revenue)")
plt.tight_layout()
plt.show()
print("CLV distribution plots rendered. Log-transformed CLV is beautifully symmetric.")

## 5. Save panel

In [ ]:
panel.to_parquet("../data/customer_panel.parquet", index=False)
print(f"Panel saved: {len(panel):,} customers, {len(panel.columns)} columns")
print("Feature engineering complete. Ready for Phase 3 (modelling).")